In [139]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta, domain_best_by_model, recommended_by_domain_for_model
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [140]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["cb06__all_fare_features"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     print(f"Experiment config: {exp_config}")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size
Experiment: fe10__fare_per_ticket_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_pclass_and_bins
Experiment: cb04__fare_and_fare_per_family
Experiment: cb05__fare_and_fare_per_ticket
Experiment: cb06__all_fare_features
Experiment: cb07__family_features
Experiment: cb08__sex_pclass_features
Experiment: ab01__age_and_bins_without_fare
Experiment: ab02__age_imputed_title_and_bins_without_fare
Experiment: ab03__age_imputed_title_pclass_and_bins_without_fare
Experiment: ab04__age_bin_without_fare
Experiment: ab05__sex_pclass_without_sibsp_parch
Experi

In [141]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [142]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

In [143]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results()
references = ['baseline__raw']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
### cb06__all_fare_features

_Description pending._

<details>
<summary>Conclusion</summary>


#### Interpretation

- Verdict: mixed
- Recommended for specific models:
  - decision_tree: test_accuracy_mean: 0.011
    - Secondary gains:
      - test_f1_mean: 0.039


#### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group           | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:------------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | cb06__all_fare_features | logreg        |                          0.786 |         

In [144]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=10, selection="all")
print("Current leaderboard:")
print(current_leaderboard)

Current leaderboard:
| experiment                                                          | model_name    |   test_accuracy_mean |   test_f1_mean |
|:--------------------------------------------------------------------|:--------------|---------------------:|---------------:|
| fe05__title__xgb                                                    | xgb           |                0.836 |          0.772 |
| ab02__age_imputed_title_and_bins_without_fare__xgb                  | xgb           |                0.835 |          0.766 |
| fe05__title__svc                                                    | svc           |                0.834 |          0.771 |
| ab03__age_imputed_title_Pclass_and_bins_without_fare__random_forest | random_forest |                0.834 |          0.762 |
| fe11__age_bin_without_fare__random_forest                           | random_forest |                0.833 |          0.759 |
| ab04__age_bin_without_fare__random_forest                           | random_fore

In [145]:
thresholds = {
    "accuracy": 0.003,
    "f1": -0.01,
}

for model in MODEL_REGISTRY:

    result = recommended_by_domain_for_model(
        results_df=all_results,
        model_name=model,
        thresholds=thresholds,
    )

    print(f"\nBest candidates for {model}")
    print("=" * 50)

    print("\nRecommended:")
    for recommendation in result["recommended"]:
        print(
            recommendation["domain"],
            "->",
            recommendation["group"],
            recommendation["deltas"],
        )
    print()
    print("recommended list:")
    for recommendation in result["recommended"]:
        print(recommendation["group"], end=", ")
    print()
    print("\nFull domain results:")
    print(result["df"].to_markdown(index=False))


Best candidates for logreg

Recommended:
title -> fe05__title {'accuracy': 0.039, 'f1': 0.052}
age -> cb03__age_imputed_title_Pclass_and_bins {'accuracy': 0.02, 'f1': 0.022}
ablation -> ab03__age_imputed_title_Pclass_and_bins_without_fare {'accuracy': 0.016, 'f1': 0.017}
sex_pclass -> fe12__sex_pclass {'accuracy': 0.016, 'f1': 0.0}
family -> fe01__family {'accuracy': 0.009, 'f1': 0.008}
cabin -> fe04__cabin_features {'accuracy': 0.005, 'f1': 0.011}
fare -> fe08__fare_per_family_member {'accuracy': 0.003, 'f1': 0.004}
ticket -> fe09__ticket_group_size {'accuracy': 0.003, 'f1': 0.003}

recommended list:
fe05__title, cb03__age_imputed_title_Pclass_and_bins, ab03__age_imputed_title_Pclass_and_bins_without_fare, fe12__sex_pclass, fe01__family, fe04__cabin_features, fe08__fare_per_family_member, fe09__ticket_group_size, 

Full domain results:
| domain     | group                                                | experiment                                                   | model_name   |   

In [146]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [147]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [148]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [149]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [150]:
# print(workflow["all_results"])

In [151]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

Old Eda practice

In [152]:
# eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print(eda)

In [153]:
# print(eda)

In [154]:
# print(eda['dataFrame_health'].to_markdown())

In [155]:
# print(eda["dataFrame_summary"].to_markdown())

In [156]:
# for col in eda['categorical_summary']:
#     sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
#     sample_df = pd.concat(
#         [sample[k] for k in ['head', 'random', 'tail']],
#         ignore_index=False
#     )
#     print(sample_df.to_markdown())
#     print()

In [157]:
# for col in eda['numerical_summary']:
#     print(f"Numerical column: {col}")
#     print(eda['numerical_summary'][col].to_markdown())
#     print()

In [158]:
# print("Correlation matrix:")
# print(eda["correlation_matrix"].to_markdown())

In [159]:
# print('Correlation with the target variable:')
# print(eda["target_correlation"].to_markdown())

In [160]:
# for col in eda["categorical_rare"]:
#     print(f"Categorical column with rare values: {col}")
#     print(eda["categorical_rare"][col])
#     print()

In [161]:
# # print(eda['sample'])
# sample_df = pd.concat(
#     [eda['sample'][k] for k in ['head', 'random', 'tail']],
#     ignore_index=True
# )
# print(sample_df.to_markdown(index=False))